# Notebook 02: DeepSAM: the COVID experiment

**Course:** Summer School on AI for Economics and Finance · ESOMAS, University of Torino (August 24–26, 2026)
**Session:** Day 2, 11:00 – 12:30: Deep Learning for Continuous-Time Models
**Slides:** `../../slides/HACT_DeepSAM_Lecture_Slides.pdf`
**Notebook role:** core (in-class walkthrough)
**Runtime:** ~2 min at `smoke`, ~6 min at `production`
**Created by:** Yucheng Yang. [Course repository](https://github.com/yangycpku/summer-school-AI-for-economics-and-finance-2026)

---


The point of solving a model with distributional feedback is that the distribution *does
something*. This notebook reproduces the three results of Section 3:

1. **Figure 1 — calibration.** The disaster state is calibrated so that the model's
   employment decline by worker type and by firm type matches what was observed in
   spring 2020.
2. **Figure 2a — recovery dynamics.** After the shock, compare the economy that is free to
   re-sort matches against one restricted to its pre-COVID composition. The gap *is* the
   distributional feedback.
3. **Figure 3 — the mechanism.** Where the feedback comes from: the acceptance sets and the
   composition of the match distribution.

Everything runs from the trained checkpoint, so nothing here trains a network.

In [ ]:
RUN_MODE = "smoke"     # one of: "smoke", "teaching", "production"

## Locating the code

`src/train_nn.py` holds the whole method: the deterministic steady states, the neural
networks, the master-equation residual, the simulation of the distribution, and the training
loop. It expects to be imported with the project root as the working directory, because
`solve_steady_state` writes its output there as `.npy` files.

On Nuvolos the notebook server starts in `/files`, which mirrors the course repository, so
the cell below changes into `/files/day2/Yang/code/DeepSAM_nuvolos`. On a local clone or
Colab it steps up from `notebooks/` to the project root instead.

In [ ]:
import os
import sys
from pathlib import Path

# On Nuvolos the kernel starts in /files, which mirrors the course repository.
NUVOLOS_ROOT = "/files/day2/Yang/code/DeepSAM_nuvolos"
if os.path.isdir(NUVOLOS_ROOT):
    os.chdir(NUVOLOS_ROOT)
elif os.path.isfile("../src/train_nn.py"):
    os.chdir("..")           # a local clone or Colab: the notebook lives in notebooks/

if not (os.path.isfile("src/train_nn.py") and os.path.isfile("config/config.yaml")):
    raise FileNotFoundError(
        f"Expected the DeepSAM project root, but the working directory is "
        f"{os.getcwd()!r}. On Nuvolos that is {NUVOLOS_ROOT!r}; elsewhere, open this "
        f"notebook from inside DeepSAM_nuvolos/notebooks."
    )

ROOT = Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

In [ ]:
import contextlib
import io
import random
import time

import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from omegaconf import OmegaConf

from train_nn import Train_NN, Master_PINN_S
import calibration_plot as calplot
import covid_shock_plot as covplot
import plotting

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
else:
    print("CPU (no GPU visible) -- everything below still runs, more slowly")

## Choosing the run mode

The costs in this notebook are all *simulation* costs: how many paths of the economy are
simulated, and for how long. `RUN_MODE` maps onto them.

| | `smoke` | `teaching` | `production` |
|---|---|---|---|
| ergodic-pool paths × horizon | 32 × 500 | 64 × 1000 | 256 × 5000 |
| pre-COVID ergodic paths | 50 | 100 | 200 |
| recovery paths averaged | 20 | 60 | 200 |
| training steps (notebook 03) | 500 | 5,000 | 20,000 |

`production` matches the settings in the replication package. Note what is *not* on this
list: training the surplus network to convergence. The full pipeline behind the shipped
checkpoint is a homotopy initialisation, a long main training phase run to a loss
threshold, and then 8 further rounds of 100,000 gradient steps with the ergodic dataset
rebuilt between rounds — several hours on an A100. That is why every notebook here starts
from the shipped checkpoint, and why notebook 03 quantifies the gap rather than trying to
close it.

In [ ]:
if RUN_MODE == "smoke":
    SIM_PATHS, SIM_T = 32, 500
    ERG_PATHS, ERG_T_END = 50, 10.0
    RECOVERY_PATHS = 20
    TRAIN_STEPS = 500
elif RUN_MODE == "teaching":
    SIM_PATHS, SIM_T = 64, 1000
    ERG_PATHS, ERG_T_END = 100, 20.0
    RECOVERY_PATHS = 60
    TRAIN_STEPS = 5_000
elif RUN_MODE == "production":
    SIM_PATHS, SIM_T = 256, 5000
    ERG_PATHS, ERG_T_END = 200, 30.0
    RECOVERY_PATHS = 200
    TRAIN_STEPS = 20_000
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")

print(
    f"RUN_MODE={RUN_MODE}: ergodic pool {SIM_PATHS}x{SIM_T}, "
    f"{ERG_PATHS} pre-COVID paths, {RECOVERY_PATHS} recovery paths"
)

## Calibration and the deterministic steady states

The calibration and training settings all live in `config/config.yaml`. We load it with
OmegaConf and pass it straight to `Train_NN`, so it is easy to see that the object is
simply built from that dictionary.

`solve_steady_state()` then solves the model's **deterministic** steady state once for each
aggregate state $z \in \{L, H, D\}$ — low, high, and the disaster state that stands in for
COVID. These are fixed points of the matching problem with the aggregate state frozen; they
are the anchors the aggregate-risk solution is built around, and the unemployment rates they
imply are the first thing to sanity-check against the calibration.

In [ ]:
cfg = OmegaConf.load(ROOT / "config" / "config.yaml")
params = {
    k: v for k, v in OmegaConf.to_container(cfg.train_nn, resolve=True).items()
    if k != "_target_"
}

seed = int(cfg.seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ct = Train_NN(**params)
print(f"device {ct.device} | {ct.nx} worker types x {ct.ny} firm types | output path {ct.path}")

t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):     # the solver is chatty; keep the summary
    ct.solve_steady_state()
print(f"solve_steady_state: {time.monotonic() - t0:.1f}s")

gm_ss = np.load("gm_ss.npy")
gm_low = np.load("gm_low_delta.npy")
gm_high = np.load("gm_high_delta.npy")
gm_dis = np.load("gm_dis_delta.npy")

# State convention (see env.py): L is the good state (separation delta_0 - d_delta),
# H the bad state (delta_0 + d_delta), D the disaster state.
for label, gm in [("baseline", gm_ss), ("low separation (L)", gm_low),
                  ("high separation (H)", gm_high), ("disaster (D)", gm_dis)]:
    u = (ct.gw.mean() - np.mean(gm)).cpu().numpy() * 100
    print(f"  unemployment rate, {label:<22s}: {u:6.3f}%")

## The trained surplus network

The one object DeepSAM learns is the **match surplus** $S(x, y, z, g)$: the value of a match
between worker type $x$ and firm type $y$, given the aggregate state $z$ *and the entire
cross-sectional distribution* $g$ of existing matches. That last argument is the hard part —
$g$ lives in $\mathbb{R}^{n_x \times n_y}$ (55 dimensions here), which is why the network
takes it directly as an input rather than summarising it.

Everything else in the model is recovered from $S$: the acceptance sets, the vacancy
posting implied by free entry, the wage through Nash bargaining, and the drift of $g$
itself.

The checkpoint below is the converged network from the paper. Notebook 03 shows what
training it involves.

In [ ]:
pinn_S = Master_PINN_S(
    nn_width=ct.nn_width,
    nn_num_layers=ct.nn_num_layers,
    n_x=ct.nx,
    n_y=ct.ny,
).to(ct.device).float()

ckpt = torch.load(ROOT / "checkpoints" / "section3_surplus_best.pt", map_location=ct.device)
pinn_S.load_state_dict(ckpt["model_state_dict"])
pinn_S.eval()

n_par = sum(p.numel() for p in pinn_S.parameters())
print(f"Loaded the trained surplus network: {ct.nn_num_layers} layers of width "
      f"{ct.nn_width}, {n_par:,} parameters")
print(f"Input dimension: 1 (x) + 1 (y) + 1 (z) + {ct.nx * ct.ny} (g) = {3 + ct.nx * ct.ny}")

## The pre-COVID economy

Before the shock hits, the economy sits in the ergodic distribution of the *two-state* $L/H$
chain — the disaster state is off. `ergodic_g_LH_ctmc` simulates that chain forward and
averages the resulting match distribution after a burn-in.

In [ ]:
t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    g0_bar, stats0, z_paths_noD = calplot.ergodic_g_LH_ctmc(
        ct=ct, pinn_S=pinn_S, g_init=None, N_paths=ERG_PATHS, dt=0.01,
        T_end=ERG_T_END, burn_in=ERG_T_END / 3, record_interval=10,
        substeps=1, clamp_g=True, seed=123,
    )
z_pre_paths = z_paths_noD[:, -1]                       # aggregate state just before COVID
z_pre_scalar = float(covplot.to_numpy(z_paths_noD)[0, -1])
print(f"pre-COVID ergodic distribution over {ERG_PATHS} paths: "
      f"{time.monotonic() - t0:.1f}s")

## Figure 1 — calibrating the disaster state

Hit the pre-COVID economy with the disaster state and let it run for 0.2 years, then compare
the employment decline by worker type and by firm type with the empirical targets from
spring 2020. The disaster separation rate $\delta(x, y)$ is what is being calibrated here.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    fig1_out = calplot.simulate_disaster_0p2_for_figure1(
        ct=ct, pinn_S=pinn_S, g0_paths=g0_bar, z_pre_paths=z_pre_paths,
        dt=0.01, T_shock=0.2, substeps=1, clamp_g=True,
    )

target_worker = np.array([0.372, 0.236, 0.180, 0.141, 0.087])
target_firm = np.array(
    [0.138, 0.012, 0.121, 0.165, 0.126, 0.177, 0.175, 0.198, 0.21, 0.289, 0.325]
)[::-1]
model_worker = fig1_out["worker_emp_drop_pct_mean"].numpy()
model_firm = fig1_out["firm_emp_drop_pct_mean"].numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=140)
for ax, model, target, name in [
    (axes[0], model_worker, target_worker, "worker type $x$"),
    (axes[1], model_firm, target_firm, "firm type $y$"),
]:
    xb = np.arange(len(model)) / (len(model) - 1)
    ax.bar(xb, model * 100, width=0.04, color="tab:blue", label="model")
    ax.bar(xb, target * 100, width=0.02, color="tab:red", label="calibration target")
    ax.set_xlabel(name)
    ax.set_ylabel("employment decline (%)")
    ax.legend(fontsize=9)
axes[0].set_title("Employment decline by worker type")
axes[1].set_title("Employment decline by firm type")
plt.tight_layout()
plt.show()

print(f"model employment decline: {model_worker[0]:.1%} for the lowest worker type, "
      f"{model_worker[-1]:.1%} for the highest (targets: "
      f"{target_worker[0]:.1%} and {target_worker[-1]:.1%})")

## Figure 2a — what the distribution does

Two economies, the same shock and the same aggregate path. One is free to re-sort: after the
disaster, new matches form wherever the surplus is positive given the *current* distribution.
The other is held to its pre-COVID composition.

The difference between the two paths is the distributional feedback — the object that a
representative-agent or a fixed-distribution treatment cannot produce.

In [ ]:
t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    res_blue = covplot.simulate_figure3a_blue_one_path(
        ct=ct, pinn_S=pinn_S, gm_low=gm_low, gm_high=gm_high, g0_bar=g0_bar,
        z_pre_scalar=z_pre_scalar, z_after_start="H", dt=0.01, T_end=2.0,
        t_covid=0.2, substeps=1, clamp_g=True, seed=123, k_mu=3, fixed_z=True,
    )
    res_orange = covplot.simulate_figure3a_orange_one_path(
        ct=ct, pinn_S=pinn_S, gm_low=gm_low, gm_high=gm_high, g0_bar=g0_bar,
        z_pre_scalar=z_pre_scalar, z_path_full=res_blue["z_path"],
        substeps=1, clamp_g=True,
    )
print(f"single-path recovery: {time.monotonic() - t0:.1f}s")
plotting.plot_relative_employment_2a(res_blue, res_orange)

### Averaging over recovery paths

One path is a story; the average over many is the result. This is the most expensive cell in
the notebook — it is what `RECOVERY_PATHS` controls.

In [ ]:
t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    final_out = covplot.plot_figure3a_average_many_paths(
        ct=ct, pinn_S=pinn_S, g0_bar=g0_bar, z_pre_scalar=z_pre_scalar,
        z_after_start="H", N_recovery_paths=RECOVERY_PATHS, dt=0.01, T_end=2.0,
        t_covid=0.2, substeps=1, clamp_g=True, seed_recovery_paths=123,
    )
print(f"{RECOVERY_PATHS} recovery paths: {time.monotonic() - t0:.1f}s")
plotting.plot_relative_employment_many(final_out)

## Figure 3 — where the feedback comes from

The acceptance sets and the composition of the match distribution, before and after the
shock. This is the mechanism behind the gap in the previous figure: the disaster destroys
matches selectively, which changes *which* new matches are worth forming, which changes the
speed of the recovery.

In [ ]:
fig4_sep = plotting.plot_figure4_panels_separately(
    ct=ct, pinn_S=pinn_S, z_ergodic_for_alpha="L",
    dpi=140, panel_ratio=800 / 543, fig_height=4.0, panels=("a", "b", "c", "d"),
)

## Summary

* The disaster state is calibrated to the observed spring-2020 employment decline by worker
  and firm type — a two-sided target that a model without two-sided heterogeneity could not
  hit.
* Letting the economy re-sort after the shock produces a materially different recovery from
  holding the match distribution fixed.
* That gap is the distributional feedback, and it is only computable because the solution
  carries $g$ in the state.

## Takeaway

The distribution is not a bookkeeping device here — it changes the aggregate path. The
methodological cost of that is having to solve a master equation in 55 state dimensions,
which is what the neural network buys you. Notebook 03 shows what paying that cost looks
like.